In [7]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

RANDOM_STATE = 42
CLASS_LABELS_MADRID    = {1: 'Pre-1960',  2: '1960-1984', 3: '1984-2004', 4: '2004-2024'}
CLASS_LABELS_AMSTERDAM = {1: 'Pre-1945',  2: '1945-1984', 3: '1984-2004', 4: '2004-2024'}

In [8]:
# 1. CARREGAR OS DADOS PREPROCESSADOS
with open('data/preprocessed/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_madrid            = data['X_madrid']
y_madrid            = data['y_madrid']
X_amsterdam         = data['X_amsterdam']
y_amsterdam         = data['y_amsterdam']
feature_names       = data['feature_names']
pixel_ids_madrid    = data['pixel_ids_madrid']
pixel_ids_amsterdam = data['pixel_ids_amsterdam']

def to_xgb_labels(y):
    return y - 1

def from_xgb_labels(y):
    return y + 1

In [9]:
# 2. HIPERPARÂMETROS OTIMIZADOS DO XGBOOST
# Corrigido: reg_alpha, reg_lambda e colsample para evitar sobreajuste à Classe 2
XGB_PARAMS = dict(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='multi:softprob',
    num_class=4,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric='mlogloss',
)

In [10]:
# ==================================================================================
# PARTE A — VALIDAÇÃO CRUZADA EM MADRID
# ==================================================================================
N_FOLDS, N_REPEATS = 5, 5
rskf = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
all_f1 = []

print(f"\n[PARTE A] Validação cruzada em Madrid — {N_FOLDS}-fold x {N_REPEATS}-repeat")
for fit_num, (tr_idx, val_idx) in enumerate(rskf.split(X_madrid, y_madrid), 1):
    X_tr, X_val = X_madrid[tr_idx], X_madrid[val_idx]
    y_tr, y_val = to_xgb_labels(y_madrid[tr_idx]), y_madrid[val_idx]

    clf = XGBClassifier(**XGB_PARAMS)
    clf.fit(X_tr, y_tr)

    y_pred = from_xgb_labels(clf.predict(X_val))
    f1 = f1_score(y_madrid[val_idx], y_pred, average='macro')
    all_f1.append(f1)

print(f"  >> Macro-F1 médio (Madrid, XGBoost Otimizado): {np.mean(all_f1):.4f} +/- {np.std(all_f1):.4f}")


[PARTE A] Validação cruzada em Madrid — 5-fold x 5-repeat
  >> Macro-F1 médio (Madrid, XGBoost Otimizado): 0.6279 +/- 0.0044


In [11]:
# PARTE B — TREINAMENTO DO MODELO FINAL
# ==================================================================================
print("\n[PARTE B] Treinando modelo final XGBoost com todos os dados de Madrid...")
model_final = XGBClassifier(**XGB_PARAMS)
model_final.fit(X_madrid, to_xgb_labels(y_madrid))

# ==================================================================================
# PARTE C — ZERO-SHOT EM AMSTERDAM
# ==================================================================================
print("\n[PARTE C] Zero-shot em Amsterdam...")
y_pred_zero = from_xgb_labels(model_final.predict(X_amsterdam))
f1_zero = f1_score(y_amsterdam, y_pred_zero, average='macro')
acc_zero = accuracy_score(y_amsterdam, y_pred_zero)

print(f"  Macro-F1 zero-shot: {f1_zero:.4f}   |  Accuracy: {acc_zero:.4f}")



[PARTE B] Treinando modelo final XGBoost com todos os dados de Madrid...

[PARTE C] Zero-shot em Amsterdam...
  Macro-F1 zero-shot: 0.5221   |  Accuracy: 0.5367


In [12]:
# ==================================================================================
# NOTEBOOK 4 (MODELLING) - CÓDIGO CORRIGIDO E OTIMIZADO
# ==================================================================================

# ==================================================================================
# PARTE D — FEW-SHOT ENSEMBLE E MAHALANOBIS TRANSFER (CORREÇÃO CRÍTICA)
# ==================================================================================
# Substituição da distância Euclidiana simples por Mahalanobis Ponderada +
# Ensemble com as probabilidades do Stage 1 (XGBoost Zero-Shot)

def mahalanobis_prototype_predict_proba(X_support, y_support, X_query, alpha=1e-3):
    classes = np.unique(y_support)
    n_features = X_support.shape[1]
    prototypes = []
    inv_covs = []

    for c in classes:
        X_c = X_support[y_support == c]
        proto = X_c.mean(axis=0)
        prototypes.append(proto)
        
        # Covariância regularizada para poucas amostras
        cov = np.cov(X_c, rowvar=False) + alpha * np.eye(n_features)
        inv_covs.append(np.linalg.pinv(cov))

    dists = np.zeros((len(X_query), len(classes)))
    for idx, c in enumerate(classes):
        diff = X_query - prototypes[idx]
        # Distância de Mahalanobis: (x-u)^T * Sigma^-1 * (x-u)
        dists[:, idx] = np.sum(np.dot(diff, inv_covs[idx]) * diff, axis=1)

    neg_d = -0.5 * dists
    neg_d -= neg_d.max(axis=1, keepdims=True)
    exp_d = np.exp(neg_d)
    return classes, exp_d / exp_d.sum(axis=1, keepdims=True)

SHOTS_PER_CLASS = [5, 10, 25, 50, 100, 200]
N_TRIALS        = 10
classes_ams     = np.unique(y_amsterdam)
rng             = np.random.default_rng(RANDOM_STATE)

# Probabilidades pré-calculadas do XGBoost Zero-Shot para o Ensemble
proba_xgb_zero  = model_final.predict_proba(X_amsterdam)

print("\n[PARTE D] Few-shot com Mahalanobis + Ensemble XGBoost em Amsterdam...")
proto_results = {n: [] for n in SHOTS_PER_CLASS}

for n_shots in SHOTS_PER_CLASS:
    for _ in range(N_TRIALS):
        support_idx = []
        for cls in classes_ams:
            cls_idx = np.where(y_amsterdam == cls)[0]
            chosen  = rng.choice(cls_idx, min(n_shots, len(cls_idx)), replace=False)
            support_idx.extend(chosen.tolist())
        
        support_idx = np.array(support_idx)
        query_mask  = np.ones(len(y_amsterdam), dtype=bool)
        query_mask[support_idx] = False

        # 1. Obter probabilidades dos protótipos adaptados
        classes, proba_proto = mahalanobis_prototype_predict_proba(
            X_amsterdam[support_idx], y_amsterdam[support_idx], X_amsterdam[query_mask]
        )

        # 2. Combinar com as probabilidades do XGBoost Zero-Shot (Peso Ponderado)
        # Dá mais peso ao XGBoost quando n_shots é muito pequeno
        weight_proto = min(0.8, n_shots / 50.0)
        weight_xgb   = 1.0 - weight_proto
        
        proba_ensemble = (weight_proto * proba_proto) + (weight_xgb * proba_xgb_zero[query_mask])
        y_pred_ensemble = classes[proba_ensemble.argmax(axis=1)]

        f1 = f1_score(y_amsterdam[query_mask], y_pred_ensemble, average='macro', zero_division=0)
        proto_results[n_shots].append(f1)

    mn, sd = np.mean(proto_results[n_shots]), np.std(proto_results[n_shots])
    print(f"  {n_shots:4d} amostras/classe: macro-F1 (Ensemble) = {mn:.4f} +/- {sd:.4f}")


[PARTE D] Few-shot com Mahalanobis + Ensemble XGBoost em Amsterdam...
     5 amostras/classe: macro-F1 (Ensemble) = 0.5440 +/- 0.0098
    10 amostras/classe: macro-F1 (Ensemble) = 0.5622 +/- 0.0157
    25 amostras/classe: macro-F1 (Ensemble) = 0.5280 +/- 0.0297
    50 amostras/classe: macro-F1 (Ensemble) = 0.4630 +/- 0.0417
   100 amostras/classe: macro-F1 (Ensemble) = 0.4193 +/- 0.0224
   200 amostras/classe: macro-F1 (Ensemble) = 0.4459 +/- 0.0266
